In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.set_option('display.max_columns', None)

## 1. General overview of data and problem statement

In [ ]:
training_data = pd.read_csv("datasets/house_prices_dataset/train.csv")

In [ ]:
training_data.head()

In [ ]:
training_data.shape

In [ ]:
training_data.info()

In [ ]:
# Verify amount of null values per column
nan_row_info = training_data.isna().sum().sort_values(ascending=True)
nan_row_info[nan_row_info > 0]

In [ ]:
# Verify amount of null values in terms of percentage
nan_row_info[nan_row_info > 0] / training_data.shape[0] *100

In [ ]:
# General info about the data behavior
training_data.describe(include=[np.int64]) # Some of them may be categorical, but expressed as integers

In [ ]:
training_data.describe(include=[str])

In [ ]:
# Verify how many columns are of dtype=str
training_data.describe(include=[str]).shape[1]

In [ ]:
# Verify Id column
training_data.Id.value_counts()

In [ ]:
# Verify uniqueness of Id column
training_data.Id.value_counts()[training_data.Id.value_counts() != 1]

In [ ]:
# Verify duplicates (without considering the null values)
print(training_data.shape)
print(training_data.drop(columns=["Id"]).drop_duplicates().shape)

### Observations
#### - 1460 rows: Probably not big enough to justify a complicated model like a neural network
#### - Fence, Alley, MiscFeature and PoolQC have more than 80% null data. These features should be verified in the EDA to determine whether to keep them or not. And if we keep some feature with null values, we must determine how to transform those null values, depending on the general behavior of the data.
#### - MasVnrType has a lower percentage of null values, but it's almost 60%. And FireplaceQu has almost 50% null values. Those features need to be verified also.
#### - For the other features with lower percentages of nulls, these need to be verified but there is a lower probability that these features may be dropped due to low amount of null values
#### - The Id column may probably be dropped, since it's basically behaving like an index with no useful information for house prices prediction.
#### - No duplicates for now, regardless of whether the Id column is considered or not. However, more verification is needed since it's possible that some rows may have very similar values and thus possibly represent the same data point.
#### - Various numerical columns have a high standard deviation relative to the mean. For example: some variables have a standard deviation higher than the mean. Others have a standard deviation close in magnitude to the mean. Therefore, histogram analysis is needed for sure.
#### - A lot of variables will have a high magnitude skew, since various variables have the min and the lower percentiles as zero, while the higher percentages have a much higher magnitude. It's possible that some nonlinear transformation is needed. But that will be determined during EDA.
#### - More than half of the features are categorical features (43) than numerical features.
#### - Some categorical features are highly skewed, since for some of them, one category appears in more than 1000 rows. Therefore, the data is imbalanced for various features. These features need to be verified. If a categorical feature is almost completely dominated by one category (example: more than 90% of the rows corresponds to one category), the model may not learn enough patterns to determine how that feature affects the target variable. Therefore, some features may be dropped during data cleaning section due to this. But more verification is needed.
#### - Some anomaly detection may be necessary, especially for highly skewed features
#### - Since there are a lot of features, we can use techniques like correlation matrix, scatter plots and clustering to determine if some features are redundant. Or we can use L1 regularization so that the model can determine which are the most important features. Or a combination of both approaches.

#### - Target variable: The mean and standard deviation are ~180,000 and ~80,000, respectively. The standard deviation is 44% of the mean, so there is a lot of fluctuation in the data. Some verification is needed to verify the skew of the target variable

## Problem statement
#### - Type of problem: 
##### -- Regression (since the target column SalePrice is a numerical column and continuous)
#### - Possible models to consider: 
##### -- Linear Regression, Random Forest Regressor, XGBoost
#### - Model Metrics:
##### -- Mean squared error
#### - Baseline performance: 
##### -- None for now
#### - Who may use a model like this: 
##### -- Customers that may be interested in buying a house. In this case, we would need to keep the model simple and possibly focus on a low amount of features to increase interpretability, since general customers may not be interested in many details.
##### -- House prices experts: In this case, keeping as many features as possible would be necessary since one may be focused on the details, but probably not too many features depending on the mathematical background that one may have.
##### -- Statisticians: In this case, keeping as many features as possibles may be useful since one may be interested in a more complex model



## 2. EDA

### General EDA

In [ ]:
training_data.head()

In [ ]:
# Let's verify null values

In [ ]:
training_data.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
training_data.isna().sum().sort_values(ascending=False).head(20) / training_data.shape[0] * 100

In [ ]:
training_data.shape[0]

### Distribution of target variable

In [ ]:
number_of_bins = int(training_data.shape[0] / 10)
number_of_bins

In [ ]:
# Distribution of target variable
training_data["SalePrice"].hist(bins=number_of_bins)
plt.xlabel("Value")
plt.show()

In [ ]:
float(training_data["SalePrice"].skew())

In [ ]:
training_data["SalePrice"].describe()

### Verify for possible outliers

In [ ]:
for value in range(10):
    percentile = 0.90 + value/100
    print(f"{percentile*100}th percentile: ", float(training_data["SalePrice"].quantile(percentile)))

In [ ]:
for value in range(10):
    percentile = 0.99 + value/1000
    print(f"{percentile*100}th percentile: ", float(training_data["SalePrice"].quantile(percentile)))

In [ ]:
training_data["SalePrice"].quantile(0.98)

### Observations

#### - The behavior of the target variable SalePrice is not perfectly gaussian. Currently it is right skewed, since 90% of the values range from 34900 to ~27800, while the remaining 10% have a wider gap between ~27800 and ~75500
#### - Therefore, a nonlinear transformation could be apply to this variable to allow the model to generalize better
#### - For now, we can use log(x+c), where c is a constant that is minimum zero. One can increase that value until one gets a good enough gaussian behavior

### Distribution of features

In [ ]:
# The majority of the features are integers. However, some of those look categorical. Let's plot the histograms to verify.
training_data.hist(bins=number_of_bins, figsize=(20,20))
plt.tight_layout()
plt.show()

In [ ]:
# Let's focus on the features that have the least amount of null values
training_data.isna().sum().sort_values(ascending=True).head(70) / training_data.shape[0] * 100

### Observations:
#### The ID has skew=0 and the values are all unique. Thus, it behaves the same as an index column. Therefore, we can remove this column.

#### The following numerical columns look categorical:
####   - OverallQual,OverallCond, BsmtFullBath, BsmtHalfBath, FullBath, HalfBath
#### MoSold looks categorical because, even if it represents a number, it is limited between 1-12.


In [ ]:
# Training data for non-numeric columns
training_data.select_dtypes(exclude=[np.number])

In [ ]:
training_data.select_dtypes(exclude=[np.number]).describe()

In [ ]:
training_data.select_dtypes(exclude=[np.number]).isna().sum()

#### Observations
##### - It's hard to determine which features are important for now. Let's perform a correlation matrix analysis

### Correlation matrix

In [ ]:
non_numeric_columns = training_data.select_dtypes(exclude=[np.number]).describe().columns.tolist()

In [ ]:
for col in non_numeric_columns:
    print(col, ":  ",  training_data[col].fillna("000").unique().tolist())

In [ ]:
for col in non_numeric_columns:
    if "000" in training_data[col].fillna("000").unique().tolist():
        print(col)

In [ ]:
# Let's use label encoding to transform the non-numeric columns

In [ ]:
# Order for label encoding (By changing nan to "000", we ensure that the LabelEncoder picks those values as the first category),
# since the categories are assigned in alphabetical order
mapping = {}
mapping["LotShape"] = ['Reg', 'IR1', 'IR2', 'IR3']
mapping["LandContour"] = ['Lvl', 'Bnk', 'Low', 'HLS']
mapping["Utilities"] = ['NoSeWa', 'AllPub']
mapping["LandSlope"] = ['Gtl', 'Mod', 'Sev']
mapping["Condition1"]= ['Norm', 'Artery', 'RRNn', 'Feedr', 'PosN', 'PosA', 'RRAn', 'RRAe', 'RRNe']
mapping["Condition2"]= ['Norm', 'Artery', 'RRNn', 'Feedr', 'PosN', 'PosA', 'RRAn', 'RRAe']
mapping["BldgType"] = ['1Fam', '2fmCon', 'Duplex', 'TwnhsE', 'Twnhs']
mapping["HouseStyle"] = ['SFoyer', 'SLvl', '1Story', '1.5Unf', '1.5Fin', '2Story', '2.5Unf', '2.5Fin']
mapping["Exterior1st"] = ['VinylSd', 'MetalSd', 'Wd Sdng', 'HdBoard', 'BrkFace', 'CemntBd', 'Plywood', 'AsbShng', 'Stucco', 'AsphShn', 'Stone', 'ImStucc', 'CBlock', 'WdShing', 'BrkComm', 'Other', 'Brk Cmn', 'Wd Shng']
mapping["Exterior2nd"] = ['VinylSd', 'MetalSd', 'Wd Sdng', 'HdBoard', 'BrkFace', 'CmentBd', 'Plywood', 'AsbShng', 'Stucco', 'AsphShn', 'Stone', 'ImStucc', 'CBlock', 'WdShing', 'BrkComm', 'Other', 'Brk Cmn', 'Wd Shng']

mapping["ExterQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["ExterCond"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["BsmtQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["BsmtCond"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["BsmtExposure"] = ['000', 'No', 'Mn', 'Av', 'Gd']
mapping["BsmtFinType1"] = ['000', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
mapping["BsmtFinType2"] = ['000', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
mapping["HeatingQC"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["CentralAir"] = ['N', 'Y']
mapping["KitchenQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["Functional"] = ['Sev', 'Sal', 'Typ', 'Min1', 'Min2', 'Mod', 'Maj1', 'Maj2']
mapping["FireplaceQu"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

mapping["GarageFinish"] = ['000', 'Unf', 'RFn', 'Fin']
mapping["GarageQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["GarageCond"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["PavedDrive"] = ['N', 'P', 'Y']
mapping["PoolQC"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
mapping["Fence"] = ['000', 'MnPrv', 'GdWo', 'MnWw', 'GdPrv']
mapping["MiscFeature"] = ['000', 'Shed', 'Gar2', 'Othr', 'TenC']



In [ ]:
training_data_transformed = training_data.copy()
for col in non_numeric_columns:
    print(f"Processing: {col}")
    if col in mapping:
        oe = OrdinalEncoder(categories=[mapping[col]])
    else:
        oe = OrdinalEncoder()
    training_data_transformed[col] = training_data_transformed[col].fillna("000")
    training_data_transformed[col] = oe.fit_transform(training_data_transformed[[col]])

In [ ]:
training_data_transformed.head()

In [ ]:
len(training_data_transformed.select_dtypes("number").columns)

In [ ]:
training_data_transformed.dtypes.unique()

In [ ]:
corr_matrix = training_data_transformed.select_dtypes("number").corr()
sns.heatmap(corr_matrix)
plt.show()

In [ ]:
# Verify correlation between target variable and the features
corr_matrix[["SalePrice"]]

In [ ]:
# Verify the features with the highest magnitude positive correlation with SalePrice
corr_matrix[["SalePrice"]].sort_values(by=["SalePrice"], ascending=False).head(20)

In [ ]:
# Verify the features with the highest magnitude negative correlation with SalePrice
corr_matrix[["SalePrice"]].sort_values(by=["SalePrice"], ascending=True).head(20)

### For now, let's choose the following features (15 for now):

#### "OverallQual": General quality is important for determining price
#### "GrLivArea": Above ground house size also is important
#### "ExterQual": External quality is important
#### "KitchenQual": Kitchen quality is important
#### "GarageCars": Many house customers may prefer garages where they can fit as much cars as possible
#### "GarageArea": We need to do an analysis between this variable and the GarageCars, since they basically represent the size of the garage. Possibly one can keep either GarageArea or GarageCars or both.
#### "TotalBsmtSF": Total basement size is important for the customers that want a basement.
#### "1stFlrSF": May need to determine if this is redundant, given that there is also "GrLivArea". But every home (or mostly every home) will have a first floor, so the size of that floor is important
#### "BsmtQual": Basement quality is important
#### "FullBath": Important since it determines how many full bathrooms have good quality
#### "GarageFinish": Garage quality is important
#### "TotRmsAbvGrd": Total rooms above grade is important
#### "YearBuilt": Some customers may prefer modern houses, while other customers may prefer older and classical style houses
#### "FireplaceQu": May be useful for customers who are interested in fireplace
#### "YearRemodAdd": The YearBuilt feature may not be useful by itself because if the house is in bad conditions, the YearBuilt won't matter much for customers. However, if the YearRemodAdd is not the same as the YearBuilt, then that means that the house had remodelation, which may be relevant since customers may be interested in houses that are modernized. Although that will depend on the quality of the remodeling and how often there was remodeling. The data description doesn't specify whether the year corresponds to the last remodeled year, the first remodeled year, etc. We can verify through the model how relevant is the feature.
#### "HeatingQC": Even if correlation value is lower, the temperature of the house can be important. Possibly, it's more important in extreme temperature conditions (example: very cold weathers or very hot weathers). However, it remains to be seen whether this feature is relevant or not, because we don't know how many of the houses in the data are built on conditions that may require a heater/thermostat. We can keep the feature for now and allow the model to decide if it's a useful feature (by using regularization).


In [ ]:
features_to_keep = ["OverallQual", "GrLivArea", "ExterQual", "KitchenQual", "GarageCars",
                    "GarageArea", "TotalBsmtSF", "1stFlrSF", "BsmtQual", "FullBath",
                    "GarageFinish", "TotRmsAbvGrd", "YearBuilt", "FireplaceQu", "YearRemodAdd",
                    "HeatingQC"]

In [ ]:
# Let's use the dataframe that has the transformations
training_data_transformed = training_data_transformed[features_to_keep + ["SalePrice"]]

In [ ]:
# Verify null values in the remaining columns
training_data_transformed.isna().sum()

In [ ]:
# Verify for duplicate rows in the remaining rows
print(training_data_transformed.shape)
print(training_data_transformed.drop_duplicates().shape)

### Histograms for the transformed dataframe

In [ ]:
training_data_transformed[features_to_keep].hist(bins=number_of_bins, 
                                                 figsize=(10,60), 
                                                 layout=(len(features_to_keep),1)
                                                )
plt.subplots_adjust()
plt.show()

In [ ]:
training_data_transformed["SalePrice"].hist(bins=number_of_bins)
plt.show()

#### Observations:
##### - Some features are right-skewed and others are left-skewed.
##### - The target variable SalePrice is right-skewed.

##### Let's verify the skew values.

In [ ]:
training_data_transformed.skew().sort_values(ascending=False)

In [ ]:
training_data_transformed.min()

In [ ]:
training_data_transformed.max()

#### Observations
##### - GarageArea has a lot of values as zero. Let's verify what those values mean by verifying the garage-related columns.

In [ ]:
training_data_transformed[training_data_transformed.GarageArea ==0]

In [ ]:
training_data_transformed[training_data_transformed.GarageCars == 0][
                                    ["GarageCars", "GarageArea", "GarageFinish"]
                                ].drop_duplicates(ignore_index=True)

In [ ]:
training_data_transformed[training_data_transformed.GarageArea == 0][
                                    ["GarageCars", "GarageArea", "GarageFinish"]
                                ].drop_duplicates(ignore_index=True)

In [ ]:
training_data_transformed[training_data_transformed.GarageFinish == 0][
                                    ["GarageCars", "GarageArea", "GarageFinish"]
                                ].drop_duplicates(ignore_index=True)

In [ ]:
training_data_transformed[training_data_transformed.GarageFinish == 0].shape[0] / training_data_transformed.shape[0] * 100

#### Observations:
##### - It seems that the three garage-related variables are consistent. If one of them are zero, then the other two are zero.
##### - This indicates that if one of the variables are zero, then there is no garage.

##### - Therefore, these zero values make sense. If a nonlinear transformation is applied to GarageArea, it needs to be one that doesn't diverge at x=0 (example: Cannot use log(x))

#### Let's do a similar verification for the basement-related features

In [ ]:
training_data_transformed[training_data_transformed.TotalBsmtSF == 0][
                                    ["TotalBsmtSF", "BsmtQual"]
                                ].drop_duplicates(ignore_index=True)

In [ ]:
training_data_transformed[training_data_transformed.BsmtQual == 0][
                                    ["TotalBsmtSF", "BsmtQual"]
                                ].drop_duplicates(ignore_index=True)

#### If one of the two above variables is zero, then there is no basement, it seems. So for now, we can keep those rows where the value is zero.

### Multicollinearity analysis

#### Correlation matrix (only for the features)

In [ ]:
training_data_transformed.columns

In [ ]:
corr_matrix = training_data_transformed[[
                                            'OverallQual', 'GrLivArea', 'ExterQual', 'KitchenQual', 'GarageCars',
                                            'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'BsmtQual', 'FullBath',
                                            'GarageFinish', 'TotRmsAbvGrd', 'YearBuilt', 'FireplaceQu',
                                            'YearRemodAdd', 'HeatingQC']].corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True)
plt.show()

#### High correlations found include the following:

##### - GarageCars - GarageArea: 0.88
##### - GrLivArea - TotRmsAbvGrd: 0.83
##### - 1stFlrSF - TotalBsmtSF: 0.82
##### - ExterQual - KitchenQual: 0.72
##### - OverallQual - ExternalQual: 0.73
##### - OverallQual - KitchenQual: 0.67

#### Perform scatter plots to verify relationship between variables

In [ ]:
plt.scatter(training_data_transformed["GarageArea"], training_data_transformed["GarageCars"])
plt.xlabel("GarageArea")
plt.ylabel("GarageCars")
plt.show()

#### Observations:
##### - No clear dependency between GarageArea and GarageCars.
##### - It seems that GarageCars doesn't always increase with GarageArea.
##### - We can keep both features for now, and allow the model to decide how important are the features.

In [ ]:
# GrLivArea - TotRmsAbvGrd
plt.scatter(training_data_transformed["GrLivArea"], training_data_transformed["TotRmsAbvGrd"])
plt.xlabel("GrLivArea")
plt.ylabel("TotRmsAbvGrd")
plt.show()

#### Observations:
##### - No clear linear dependency either. We can keep both features for now.

In [ ]:
# 1stFlrSF - TotalBsmtSF
plt.scatter(training_data_transformed["1stFlrSF"], training_data_transformed["TotalBsmtSF"])
plt.xlabel("1stFlrSF")
plt.ylabel("TotalBsmtSF")
plt.show()

#### Observations:
##### - No clear linear dependency either. We can keep both features for now.

In [ ]:
# ExterQual - KitchenQual
plt.scatter(training_data_transformed["ExterQual"], training_data_transformed["KitchenQual"])
plt.xlabel("ExterQual")
plt.ylabel("KitchenQual")
plt.show()

#### Observations:
##### - No clear linear dependency either. We can keep both features for now.

In [ ]:
# OverallQual - ExterQual
plt.scatter(training_data_transformed["OverallQual"], training_data_transformed["ExterQual"])
plt.xlabel("OverallQual")
plt.ylabel("ExterQual")
plt.show()

#### Observations:
##### - No clear linear dependency either. We can keep both features for now.

In [ ]:
# OverallQual - KitchenQual
plt.scatter(training_data_transformed["OverallQual"], training_data_transformed["KitchenQual"])
plt.xlabel("OverallQual")
plt.ylabel("KitchenQual")
plt.show()

#### Observations:
##### - No clear linear dependency either. We can keep both features for now.

## 3. Data cleaning

### Data cleaning for training data

In [ ]:
training_data.head()

In [ ]:
def keep_relevant_cols(df: pd.Dataframe) -> pd.Dataframe:
    return df[features_to_keep + ["SalePrice"]]

def encode_non_numeric_cols(df: pd.Dataframe) -> pd.Dataframe:
    final_mapping = {}
    final_mapping["BsmtQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
    final_mapping["GarageFinish"] = ['000', 'Unf', 'RFn', 'Fin']
    final_mapping["FireplaceQu"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
    final_mapping["KitchenQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
    final_mapping["ExterQual"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
    final_mapping["HeatingQC"] = ['000', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
    for col in final_mapping.keys():
        print(f"Processing: {col}")
        if col in final_mapping:
            oe = OrdinalEncoder(categories=[final_mapping[col]])
        else:
            oe = OrdinalEncoder()
        df[col] = df[col].fillna("000")
        df[col] = oe.fit_transform(df[[col]])
    return df

In [ ]:
training_data_cleaned = keep_relevant_cols(training_data)
training_data_cleaned = encode_non_numeric_cols(training_data_cleaned)

In [ ]:
training_data_cleaned.head()

### Histograms

In [ ]:
# Features
training_data_cleaned[features_to_keep].hist(bins=number_of_bins, 
                           figsize=(10,60), 
                           layout=(len(features_to_keep),1)
                          )
plt.subplots_adjust()
plt.show()

In [ ]:
# Target variable
training_data_cleaned["SalePrice"].hist(bins=number_of_bins)
plt.subplots_adjust()
plt.show()

In [ ]:
training_data_cleaned.skew().sort_values()

#### Observations:
##### - Even if GarageArea has a low skew, there seems to be a lot of values close to zero
##### - YearRemodAdd has a lot of values near 1950, so this may need verification
##### - YearBuilt doesn't look evenly distributed, even if the skew is low

##### - We can apply nonlinear transformation to the following: SalePrice, TotalBsmtSF, 1stFlrSF, GrLivArea, YearRemodAdd, YearBuilt

In [ ]:
training_data_cleaned.min().sort_values()

### Nonlinear transformation

In [ ]:
def apply_log_x_transform(df: pd.Dataframe) -> pd.Dataframe:
    for col in ["SalePrice", "1stFlrSF", "GrLivArea", "YearRemodAdd", "YearBuilt"]:
        df[col] = np.log(df[col])
    return df

def apply_log_x_plus_c_transform(df: pd.Dataframe) -> pd.Dataframe:
    c = 1
    col = "TotalBsmtSF"
    df[col] = np.log(df[col] + 1)
    return df

In [ ]:
training_data_cleaned = apply_log_x_transform(training_data_cleaned)
training_data_cleaned = apply_log_x_plus_c_transform(training_data_cleaned)

In [ ]:
# Let's verify how the histograms changed after nonlinear transform

In [ ]:
training_data_cleaned[["SalePrice", "TotalBsmtSF", "1stFlrSF", "GrLivArea", "YearRemodAdd", "YearBuilt"]
                                ].hist(bins=number_of_bins,
                                       figsize=(10,60), 
                                       layout=(len(features_to_keep),1)
                                      )
plt.subplots_adjust()
plt.show()

In [ ]:
training_data_cleaned.skew()

#### Observations:
##### There seem to be outliers in TotalBsmtSF and YearRemodAdd. We can keep them for now and verify how the model performs.

## 4. Model

In [ ]:
training_data_cleaned.head()

In [ ]:
training_data_cleaned.shape[0]

#### Note:
##### - For now, since the test data is unlabeled, we will use the same training data to create the validation set and the test set. Once test data is available with labels, then that test data will be used to create the validation set and the test set (since the validation set and test set must reflect real-world scenarios, if possible) 

In [ ]:
# 60% training, 20% validation, 20% test

X_train, X_val_test, y_train, y_val_test = train_test_split(training_data_cleaned[features_to_keep], 
                                                            training_data_cleaned["SalePrice"],
                                                            test_size = 0.4, 
                                                            random_state=5)

X_val, X_test, y_val, y_test = train_test_split(X_val_test, 
                                                y_val_test,
                                                test_size = 0.5, 
                                                random_state=10)

In [ ]:
# Verify if z-score (if needed)

In [ ]:
training_data_cleaned.max() - training_data_cleaned.min()

In [ ]:
# We can apply z-score to GarageArea for now

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_with_z_score = X_train.copy()
X_train_with_z_score["GarageArea"] = scaler.fit_transform(pd.DataFrame(X_train["GarageArea"]))

In [ ]:
X_train_with_z_score["GarageArea"].max() - X_train_with_z_score["GarageArea"].min()

### Linear regression

In [ ]:
# Train model

In [ ]:
model = LinearRegression()
model.fit(X_train_with_z_score, y_train);

In [ ]:
model.coef_

In [ ]:
list(model.coef_).index(max(model.coef_))

In [ ]:
features_to_keep[12]

In [ ]:
features_to_keep[14]

#### Observations:
##### - It seems that YearBuilt and YearRemodAdd have the highest weight parameters.
##### - However, additional EDA is probably needed to analyze correlation between these variables 

In [ ]:
model.intercept_

In [ ]:
# Training error

In [ ]:
y_train_pred = model.predict(X_train_with_z_score)

In [ ]:
mse_train = mean_squared_error(y_train, y_train_pred)
mse_train

In [ ]:
# Validation

In [ ]:
X_val_with_z_score = X_val.copy()
X_val_with_z_score["GarageArea"] = scaler.transform(pd.DataFrame(X_val["GarageArea"]))

In [ ]:
y_val_pred = model.predict(X_val_with_z_score)

In [ ]:
mse_val = mean_squared_error(y_val, y_val_pred)
mse_val

#### Observations:
##### -  The error is similar to the training set

In [ ]:
# Test

In [ ]:
X_test_with_z_score = X_test.copy()
X_test_with_z_score["GarageArea"] = scaler.transform(pd.DataFrame(X_test["GarageArea"]))

In [ ]:
y_test_pred = model.predict(X_test_with_z_score)

In [ ]:
mse_test = mean_squared_error(y_test, y_test_pred)
mse_test

In [ ]:
# Lowest value of log(SalePrice)
y_test.min()

In [ ]:
# Highest value of log(SalePrice)
y_test.max()

#### Observations:
##### - Error is higher in test set, but very small relative to the magnitude of the transformed target variable, which is log(SalePrice)
##### - The very low magnitude in error is due to nonlinear transformation applied to the target variable, since the log transformation can map the target variable to lower values and create a more uniform distribution
##### - To get the error in terms of the original units of SalePrice, one would need to apply an inverse transformation to the predictions (exponential transformation)


### Random forest regressor

#### Note:
##### -No z-score transformation will be applied here, since tree-based models are not affected by the magnitude of the features. They make decisions on value thresholds, not on distances/magnitudes between data points.

In [ ]:
len(features_to_keep)

In [ ]:
# Using max_depth=3 and max_features=8 to reduce the risk of overfitting

In [ ]:
rf_model = RandomForestRegressor(n_estimators=128, 
                                 criterion="squared_error",
                                 max_features=8,
                                 max_depth = 3,
                                 )

In [ ]:
rf_model.fit(X_train, y_train);

In [ ]:
# Training error

In [ ]:
y_train_pred_rf = rf_model.predict(X_train)
mse_train_rf = mean_squared_error(y_train, y_train_pred_rf)
mse_train_rf

In [ ]:
# Validation

In [ ]:
y_val_pred_rf = rf_model.predict(X_val)
mse_val_rf = mean_squared_error(y_val, y_val_pred_rf)
mse_val_rf

In [ ]:
# Test

In [ ]:
y_test_pred_rf = rf_model.predict(X_test)
mse_test_rf = mean_squared_error(y_test, y_test_pred_rf)
mse_test_rf

#### Observations:
##### - Error is higher than in linear regression, especially the training error.
##### - The model seem to be biased in comparison to linear regression training model. Possibly, this could be caused by strict values chosen for max_features and max_depth.
##### - The model is also presenting overfitting, based on the test error. Various ways to verify this include keeping less features, adding more training data and adjust model parameters to give the model more flexibility to be more complex (example: regularization)


### XGBoost

#### Note: 
##### - No z-score transformation will be applied here, since tree-based models are not affected by the magnitude of the features. They make decisions on value thresholds, not on distances/magnitudes between data points

In [ ]:
# For better comparison with random forest model, let's use the same hyperparameter values for:
#     n_estimators, 
#     objective/criterion
#     max_depth
#     max_features/colsample_bytree

In [ ]:
colsample_bytree = 8 / len(features_to_keep)
colsample_bytree

In [ ]:
xgb_model = XGBRegressor(n_estimators=128,
                         objective="reg:squarederror",
                         colsample_bytree=colsample_bytree,
                         max_depth=3)

In [ ]:
xgb_model.fit(X_train, y_train);

In [ ]:
y_train_pred_xgb = xgb_model.predict(X_train)
y_val_pred_xgb = xgb_model.predict(X_val)
y_test_pred_xgb = xgb_model.predict(X_test)

In [ ]:
print("Training MSE: ", mean_squared_error(y_train, y_train_pred_xgb))
print("Validation MSE: ", mean_squared_error(y_val, y_val_pred_xgb))
print("Test MSE: ", mean_squared_error(y_test, y_test_pred_xgb))

#### Observations
##### - The xgboost regressor model has the lowest training MSE. 
##### - In terms of validation and test error, it performs slightly worse than linear regression model, but better than the random forest model.

#### Future parameters to modify:
##### - Reduce subsample to a value less than 1: This will force each tree to use a subsample of data instead of the full training data.
##### -  Adjust learning rate.
##### -Verify if a relevant non-modified hyperparameter has a different default value in random forest model vs xgboost regressor model.

## Conclusion

##### - The random forest model had the worst performance thus far in terms of mean squared error.
##### - There is not much difference between the xgboost regressor model error and the linear regression model.
##### - However, since the xgboost regressor model has the lowest training error, we can conclude for now that the model with the best performance was the xgboost regressor model.
##### - Reminder: The MSE values are in log scale due to the log transformation applied to the target variable SalePrice. To analyze error in terms of original SalePrice units (dollars), one would need to apply an exponential transformation to the predictions.

#### Future work includes:
##### - More hyperparameter tuning, including adjusting regularization.
##### - Verify models performance when less features are used.
##### - Verify models performance when more features are used.
##### - Use additional metrics like R2 score.